# FID comparison: base inpainting vs base + LoRA

This notebook runs the same masked inpainting inputs through two variants of `stable-diffusion-v1-5/stable-diffusion-inpainting`:

- the base model with no LoRA adapter enabled
- the base model with one selected LoRA adapter loaded

It then computes FID for both generated image sets against the original dataset images. Lower FID is better. For a reliable score, increase `NUM_SAMPLES`; the small defaults are meant to keep a first run fast.


This notebook also evaluates:

- masked-region quality as a function of mask size,
- whether the LoRA adapter improves the masked region relative to base,
- and whether LoRA changes are concentrated inside the masked area versus the preserved background.


In [ ]:
from __future__ import annotations

import json
import math
import shutil
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image, ImageDraw
from scipy import linalg
from torchvision.models import Inception_V3_Weights, inception_v3


def find_repo_root(start: Path) -> Path:
    """Walk upward until the repository root is found, so the notebook works from any cwd."""
    for path in (start, *start.parents):
        if (
            (path / "backend").exists()
            and (path / "frontend").exists()
            and (path / "notebooks").exists()
        ):
            return path
    raise RuntimeError(
        "Could not find repository root containing backend/, frontend/, and notebooks/."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
REPO_ROOT

In [ ]:
# -----------------------------
# Experiment configuration
# -----------------------------

# Comment in exactly one evaluation block below.

# Detailed-image LoRAs: small tree/house sets and full houses/trees sets.
# Pulls NUM_SAMPLES images from this prepared dataset and creates random test masks.
EVALUATION_SOURCE = "detailed_random_masks"
DATASET_DIR = REPO_ROOT / "datasets" / "small_house_set"
TEST_MASK_DIR = None

# Sentinel-water LoRA: use the held-out Sentinel test masks already on disk.
# EVALUATION_SOURCE = "sentinel_test_masks"
# DATASET_DIR = REPO_ROOT / "datasets" / "sentinel_water"
# TEST_MASK_DIR = DATASET_DIR / "test_masks"

# Pick the LoRA adapter directory to compare against the base model.
# This can point at a final adapter folder or a checkpoint folder that contains pytorch_lora_weights.safetensors.
LORA_DIR = REPO_ROOT / "notebooks" / "outputs" / "lora_small_house_set_sd15_inpaint"
LORA_ADAPTER_NAME = "selected_lora"
LORA_SCALE = 1.0

# Use the same base model and generation settings as the backend pipeline.
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"
RESOLUTION = 512
NUM_INFERENCE_STEPS = 40
GUIDANCE_SCALE = 6.5
STRENGTH = 1.0

# Edit these prompts to match the LoRA being evaluated.
PROMPT = "satellite view of house, highly detailed, realistic"
NEGATIVE_PROMPT = """
blurry, distorted, repeated roofs, warped perspective, low quality
"""
# FID is noisy for tiny batches. The default keeps the first run quick on the small example dataset.
NUM_SAMPLES = 4
SEED = 7
BATCH_SIZE = 8

# Random masks used only by the detailed-image evaluation block.
RANDOM_MASK_MIN_AREA = 0.08
RANDOM_MASK_MAX_AREA = 0.22

# Keep this False when you want missing files to be fetched from Hugging Face
# Set it True only for offline runs.
LOCAL_FILES_ONLY = False

# Generated images and the score JSON are written here. The directory is git-ignored with other notebook outputs.
OUTPUT_DIR = REPO_ROOT / "notebooks" / "outputs" / "fid_base_vs_lora"
RANDOM_MASK_DIR = OUTPUT_DIR / "random_test_masks"

print(f"Evaluation: {EVALUATION_SOURCE}")
print(f"Dataset:   {DATASET_DIR}")
print(f"Test masks: {TEST_MASK_DIR}")
print(f"LoRA:      {LORA_DIR}")
print(f"Output:    {OUTPUT_DIR}")

In [ ]:
@dataclass(frozen=True)
class InpaintPair:
    image_path: Path
    mask_path: Path


IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"}


def discover_image_mask_pairs(
    dataset_dir: Path, mask_dir: Path | None = None
) -> list[InpaintPair]:
    """Find source images and their evaluation masks in the local dataset conventions used by this repo."""
    if not dataset_dir.exists():
        raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")
    if mask_dir is not None and not mask_dir.exists():
        raise FileNotFoundError(f"Mask directory not found: {mask_dir}")

    pairs: list[InpaintPair] = []
    mask_dirs = (
        [mask_dir]
        if mask_dir is not None
        else [
            dataset_dir / "masks",
            dataset_dir / "mask",
            dataset_dir / "test_mask",
            dataset_dir / "test_masks",
        ]
    )

    for image_path in sorted(dataset_dir.rglob("*")):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_SUFFIXES:
            continue

        # Skip mask files while scanning for source images.
        lowered_parts = {part.lower() for part in image_path.parts}
        if {"mask", "masks", "test_mask", "test_masks"} & lowered_parts:
            continue
        if image_path.stem.lower().endswith(("_mask", "_testmask")):
            continue

        # Match common local naming patterns: image.jpg -> masks/image_mask.png or test_masks/image_testmask.png.
        candidates = []
        for mask_dir in mask_dirs:
            candidates.extend(
                [
                    mask_dir / f"{image_path.stem}_testmask.png",
                    mask_dir / f"{image_path.stem}_testmask.jpg",
                    mask_dir / f"{image_path.stem}_mask.png",
                    mask_dir / f"{image_path.stem}_mask.jpg",
                    mask_dir / f"{image_path.stem}.png",
                    mask_dir / f"{image_path.stem}.jpg",
                ]
            )

        mask_path = next((path for path in candidates if path.exists()), None)
        if mask_path is not None:
            pairs.append(InpaintPair(image_path=image_path, mask_path=mask_path))

    # Also support the repo's single-image sanity-test layout: test_mask/image.png + test_mask/mask.png.
    if mask_dir is None:
        test_image = dataset_dir / "test_mask" / "image.png"
        test_mask = dataset_dir / "test_mask" / "mask.png"
        if test_image.exists() and test_mask.exists():
            pairs.append(InpaintPair(image_path=test_image, mask_path=test_mask))

    if not pairs:
        mask_source = f" using masks from {mask_dir}" if mask_dir is not None else ""
        raise RuntimeError(
            f"No image/mask pairs found under {dataset_dir}{mask_source}"
        )

    return pairs


def discover_dataset_images(dataset_dir: Path) -> list[Path]:
    """Find source images from a prepared dataset, preferring metadata.jsonl when present."""
    metadata_path = dataset_dir / "metadata.jsonl"
    if metadata_path.exists():
        image_paths = []
        for line in metadata_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            image_path = dataset_dir / row["image"]
            if image_path.exists():
                image_paths.append(image_path)
        if image_paths:
            return image_paths

    image_paths = []
    for image_path in sorted(dataset_dir.rglob("*")):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        lowered_parts = {part.lower() for part in image_path.parts}
        if {"mask", "masks", "test_mask", "test_masks"} & lowered_parts:
            continue
        if image_path.stem.lower().endswith(("_mask", "_testmask")):
            continue
        image_paths.append(image_path)
    if not image_paths:
        raise RuntimeError(f"No source images found under {dataset_dir}")
    return image_paths


def random_inpaint_mask(
    image_size: tuple[int, int], rng: np.random.Generator
) -> Image.Image:
    """Create one random white repaint region over a black background."""
    width, height = image_size
    target_area = (
        rng.uniform(RANDOM_MASK_MIN_AREA, RANDOM_MASK_MAX_AREA) * width * height
    )
    aspect = rng.uniform(0.65, 1.75)
    mask_width = int(np.clip(round(math.sqrt(target_area * aspect)), 32, width * 0.8))
    mask_height = int(
        np.clip(round(target_area / max(mask_width, 1)), 32, height * 0.8)
    )
    left = int(rng.integers(0, max(1, width - mask_width + 1)))
    top = int(rng.integers(0, max(1, height - mask_height + 1)))
    right = min(width, left + mask_width)
    bottom = min(height, top + mask_height)

    mask = Image.new("L", image_size, 0)
    draw = ImageDraw.Draw(mask)
    if rng.random() < 0.5:
        draw.ellipse((left, top, right, bottom), fill=255)
    else:
        radius = max(8, min(mask_width, mask_height) // 5)
        draw.rounded_rectangle((left, top, right, bottom), radius=radius, fill=255)
    return mask


def build_detailed_random_mask_pairs(
    dataset_dir: Path, mask_dir: Path, count: int, seed: int
) -> list[InpaintPair]:
    """Sample detailed dataset images and write reproducible random masks for evaluation."""
    image_paths = discover_dataset_images(dataset_dir)
    if len(image_paths) < count:
        raise ValueError(
            f"Requested {count} samples, but only found {len(image_paths)} images in {dataset_dir}"
        )

    rng = np.random.default_rng(seed)
    selected_indices = rng.choice(len(image_paths), size=count, replace=False)
    mask_dir.mkdir(parents=True, exist_ok=True)

    pairs = []
    for output_index, image_index in enumerate(selected_indices):
        image_path = image_paths[int(image_index)]
        with Image.open(image_path) as image:
            mask = random_inpaint_mask(image.size, rng)
        mask_path = mask_dir / f"{output_index:04d}_{image_path.stem}_random_mask.png"
        mask.save(mask_path)
        pairs.append(InpaintPair(image_path=image_path, mask_path=mask_path))
    return pairs


if EVALUATION_SOURCE == "sentinel_test_masks":
    all_pairs = discover_image_mask_pairs(DATASET_DIR, TEST_MASK_DIR)
    pairs = all_pairs[:NUM_SAMPLES]
elif EVALUATION_SOURCE == "detailed_random_masks":
    TEST_MASK_DIR = RANDOM_MASK_DIR
    pairs = build_detailed_random_mask_pairs(
        DATASET_DIR, RANDOM_MASK_DIR, NUM_SAMPLES, SEED
    )
    all_pairs = pairs
else:
    raise ValueError(f"Unknown EVALUATION_SOURCE: {EVALUATION_SOURCE}")

print(f"Found {len(all_pairs)} image/test-mask pairs; using {len(pairs)} for this run.")
pairs[:3]

In [ ]:
def prepare_dirs(output_dir: Path) -> dict[str, Path]:
    """Create clean output folders so each run compares exactly the images it just generated."""
    dirs = {
        "real": output_dir / "real",
        "base": output_dir / "base",
        "lora": output_dir / "lora",
    }
    for path in dirs.values():
        if path.exists():
            shutil.rmtree(path)
        path.mkdir(parents=True, exist_ok=True)
    return dirs


def load_image(path: Path) -> Image.Image:
    """Load and resize an RGB source image to the resolution expected by SD 1.5 inpainting."""
    return (
        Image.open(path)
        .convert("RGB")
        .resize((RESOLUTION, RESOLUTION), Image.Resampling.BILINEAR)
    )


def load_mask(path: Path) -> Image.Image:
    """Load the mask and force the notebook convention: white pixels are repainted."""
    mask = (
        Image.open(path)
        .convert("L")
        .resize((RESOLUTION, RESOLUTION), Image.Resampling.NEAREST)
    )
    return mask.point(lambda value: 255 if value > 127 else 0)


run_dirs = prepare_dirs(OUTPUT_DIR)

# Store the reference images used for FID next to the generated outputs for traceability.
for index, pair in enumerate(pairs):
    load_image(pair.image_path).save(run_dirs["real"] / f"{index:04d}.png")

print(run_dirs)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# The pipeline is loaded once; the adapter stays loaded and is switched between
# weight 0.0 (base behavior) and LORA_SCALE (base + LoRA behavior).
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None,
    local_files_only=LOCAL_FILES_ONLY,
    # This SD 1.5 inpainting snapshot commonly ships UNet/VAE weights as .bin files.
    # Requesting them explicitly avoids the noisy safetensors lookup warning.
    use_safetensors=False,
)

if device == "cuda":
    # CPU offload matches the backend and lowers VRAM pressure while still using CUDA for inference.
    pipe.enable_model_cpu_offload()
else:
    pipe.to(device)

if not LORA_DIR.exists():
    raise FileNotFoundError(f"LoRA directory not found: {LORA_DIR}")

# load_lora_weights accepts the diffusers layout saved by the training notebooks.
if LORA_DIR.is_dir():
    pipe.load_lora_weights(str(LORA_DIR), adapter_name=LORA_ADAPTER_NAME)
else:
    pipe.load_lora_weights(
        str(LORA_DIR.parent), weight_name=LORA_DIR.name, adapter_name=LORA_ADAPTER_NAME
    )
print(f"Pipeline loaded on {device}; LoRA adapter '{LORA_ADAPTER_NAME}' is available.")

In [ ]:
def select_base_model() -> None:
    """Use the loaded adapter at zero weight, which matches the unmodified base model."""
    pipe.enable_lora()
    pipe.set_adapters([LORA_ADAPTER_NAME], adapter_weights=[0.0])


def select_lora_model() -> None:
    """Enable the selected LoRA adapter at the configured scale."""
    pipe.enable_lora()
    pipe.set_adapters([LORA_ADAPTER_NAME], adapter_weights=[LORA_SCALE])


def inpaint_batch(output_dir: Path, use_lora: bool) -> list[Path]:
    """Run one full batch through either the base model or the LoRA-augmented model."""
    if use_lora:
        select_lora_model()
    else:
        select_base_model()

    generated_paths: list[Path] = []
    for index, pair in enumerate(pairs):
        image = load_image(pair.image_path)
        mask = load_mask(pair.mask_path)

        if float((np.asarray(mask) > 0).mean()) == 0:
            raise ValueError(f"Mask is empty after thresholding: {pair.mask_path}")

        # Use the same seed for the matching base and LoRA sample so model choice is the main difference.
        generator = torch.Generator(device).manual_seed(SEED + index)
        result = pipe(
            prompt=PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            image=image,
            mask_image=mask,
            height=RESOLUTION,
            width=RESOLUTION,
            strength=STRENGTH,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        ).images[0]

        out_path = output_dir / f"{index:04d}.png"
        result.save(out_path)
        generated_paths.append(out_path)
        print(f"Saved {out_path.name} ({'LoRA' if use_lora else 'base'})")

    return generated_paths


base_paths = inpaint_batch(run_dirs["base"], use_lora=False)
lora_paths = inpaint_batch(run_dirs["lora"], use_lora=True)
real_paths = sorted(run_dirs["real"].glob("*.png"))

# A same-seed base/LoRA comparison should not be byte-identical when the adapter is active.
base_lora_diffs = []
for base_path, lora_path in zip(base_paths, lora_paths):
    base_array = np.asarray(Image.open(base_path).convert("RGB"), dtype=np.int16)
    lora_array = np.asarray(Image.open(lora_path).convert("RGB"), dtype=np.int16)
    base_lora_diffs.append(float(np.abs(base_array - lora_array).mean()))

print("base vs LoRA mean abs diff per sample:", base_lora_diffs)
if base_lora_diffs and max(base_lora_diffs) == 0.0:
    raise RuntimeError(
        "Base and LoRA outputs are identical; check LORA_DIR, LORA_SCALE, and adapter loading."
    )

## Mask-size quality analysis

This section measures masked-region quality for the base model and the base+LoRA model, then aggregates results across five mask-size bins. It also reports whether LoRA gains are concentrated inside the mask region and how often LoRA improves masked-region PSNR.


In [ ]:
import math
from collections import defaultdict

MASK_BUCKET_EDGES = [0.0, 0.05, 0.15, 0.30, 0.50, 1.01]
MASK_BUCKET_LABELS = ["<5%", "5-15%", "15-30%", "30-50%", "50%+"]


def bucket_name(mask_pct: float) -> str:
    for left, right, label in zip(
        MASK_BUCKET_EDGES, MASK_BUCKET_EDGES[1:], MASK_BUCKET_LABELS
    ):
        if left <= mask_pct < right:
            return label
    return MASK_BUCKET_LABELS[-1]


def masked_region_stats(
    generated_path: Path, real_path: Path, mask_path: Path
) -> dict[str, float]:
    generated = np.asarray(Image.open(generated_path).convert("RGB"), dtype=np.float32)
    real = np.asarray(Image.open(real_path).convert("RGB"), dtype=np.float32)
    mask = np.asarray(load_mask(mask_path), dtype=np.uint8) > 0
    if mask.sum() == 0:
        raise ValueError(f"Mask is empty after resizing: {mask_path}")

    mask_3d = np.broadcast_to(mask[..., None], generated.shape)
    diff = np.abs(generated - real)

    masked_diff = diff[mask_3d]
    unmasked_diff = diff[~mask_3d]

    masked_mae = float(masked_diff.mean())
    masked_mse = float(((generated - real)[mask_3d] ** 2).mean())
    masked_psnr = (
        float(20.0 * math.log10(255.0) - 10.0 * math.log10(masked_mse))
        if masked_mse > 0
        else float("inf")
    )
    unmasked_mae = float(unmasked_diff.mean()) if unmasked_diff.size else 0.0

    return {
        "mask_pct": float(mask.mean()),
        "masked_mae": masked_mae,
        "masked_psnr": masked_psnr,
        "unmasked_mae": unmasked_mae,
    }


analysis_rows: list[dict[str, object]] = []
for index, pair in enumerate(pairs):
    real_path = run_dirs["real"] / f"{index:04d}.png"
    base_path = run_dirs["base"] / f"{index:04d}.png"
    lora_path = run_dirs["lora"] / f"{index:04d}.png"

    base_stats = masked_region_stats(base_path, real_path, pair.mask_path)
    lora_stats = masked_region_stats(lora_path, real_path, pair.mask_path)

    analysis_rows.append(
        {
            "index": index,
            "mask_pct": base_stats["mask_pct"],
            "bucket": bucket_name(base_stats["mask_pct"]),
            "base_masked_mae": base_stats["masked_mae"],
            "lora_masked_mae": lora_stats["masked_mae"],
            "base_masked_psnr": base_stats["masked_psnr"],
            "lora_masked_psnr": lora_stats["masked_psnr"],
            "base_unmasked_mae": base_stats["unmasked_mae"],
            "lora_unmasked_mae": lora_stats["unmasked_mae"],
        }
    )

bucket_summary: dict[str, dict[str, float]] = {
    label: {
        "count": 0,
        "base_masked_psnr": 0.0,
        "lora_masked_psnr": 0.0,
        "base_masked_mae": 0.0,
        "lora_masked_mae": 0.0,
    }
    for label in MASK_BUCKET_LABELS
}

for row in analysis_rows:
    bucket = row["bucket"]
    summary = bucket_summary[bucket]
    summary["count"] += 1
    summary["base_masked_psnr"] += row["base_masked_psnr"]
    summary["lora_masked_psnr"] += row["lora_masked_psnr"]
    summary["base_masked_mae"] += row["base_masked_mae"]
    summary["lora_masked_mae"] += row["lora_masked_mae"]

print("Mask-size bucket summary")
print("bucket,count,base_psnr,lora_psnr,base_mae,lora_mae,psnr_delta,mae_delta")
for bucket in MASK_BUCKET_LABELS:
    summary = bucket_summary[bucket]
    count = summary["count"]
    if count == 0:
        continue
    print(
        f"{bucket},{count},"
        f"{summary['base_masked_psnr'] / count:.2f},"
        f"{summary['lora_masked_psnr'] / count:.2f},"
        f"{summary['base_masked_mae'] / count:.2f},"
        f"{summary['lora_masked_mae'] / count:.2f},"
        f"{(summary['lora_masked_psnr'] - summary['base_masked_psnr']) / count:.2f},"
        f"{(summary['lora_masked_mae'] - summary['base_masked_mae']) / count:.2f}"
    )

psnr_improvements = [
    row["lora_masked_psnr"] - row["base_masked_psnr"] for row in analysis_rows
]
mae_improvements = [
    row["base_masked_mae"] - row["lora_masked_mae"] for row in analysis_rows
]
mask_leq_5 = [row for row in analysis_rows if row["mask_pct"] < 0.05]

print()
print(
    f"LoRA improved masked PSNR on {sum(1 for d in psnr_improvements if d > 0)} / {len(psnr_improvements)} samples"
)
print(
    f"LoRA improved masked MAE on {sum(1 for d in mae_improvements if d > 0)} / {len(mae_improvements)} samples"
)
print(
    f"Average unmasked MAE: base={np.mean([row['base_unmasked_mae'] for row in analysis_rows]):.3f}, "
    f"LoRA={np.mean([row['lora_unmasked_mae'] for row in analysis_rows]):.3f}"
)

plt.figure(figsize=(10, 4))
for label, color in [("base", "tab:blue"), ("lora", "tab:orange")]:
    y = [row[f"{label}_masked_psnr"] for row in analysis_rows]
    x = [row["mask_pct"] * 100.0 for row in analysis_rows]
    plt.scatter(x, y, label=f"{label} masked PSNR", alpha=0.75, color=color)

for label, key in [("base", "base_masked_psnr"), ("lora", "lora_masked_psnr")]:
    xs, ys = [], []
    for bucket in MASK_BUCKET_LABELS:
        summary = bucket_summary[bucket]
        if summary["count"] == 0:
            continue
        xs.append(
            (
                MASK_BUCKET_EDGES[MASK_BUCKET_LABELS.index(bucket)]
                + MASK_BUCKET_EDGES[MASK_BUCKET_LABELS.index(bucket) + 1]
            )
            * 50.0
        )
        ys.append(summary[key] / summary["count"])
    plt.plot(xs, ys, label=f"{label} mean", linewidth=2)

plt.xlabel("Mask area (%)")
plt.ylabel("Masked-region PSNR")
plt.title("Masked-region PSNR vs mask size")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
for label, color in [("base", "tab:blue"), ("lora", "tab:orange")]:
    y = [row[f"{label}_masked_mae"] for row in analysis_rows]
    x = [row["mask_pct"] * 100.0 for row in analysis_rows]
    plt.scatter(x, y, label=f"{label} masked MAE", alpha=0.75, color=color)

for label, key in [("base", "base_masked_mae"), ("lora", "lora_masked_mae")]:
    xs, ys = [], []
    for bucket in MASK_BUCKET_LABELS:
        summary = bucket_summary[bucket]
        if summary["count"] == 0:
            continue
        xs.append(
            (
                MASK_BUCKET_EDGES[MASK_BUCKET_LABELS.index(bucket)]
                + MASK_BUCKET_EDGES[MASK_BUCKET_LABELS.index(bucket) + 1]
            )
            * 50.0
        )
        ys.append(summary[key] / summary["count"])
    plt.plot(xs, ys, label=f"{label} mean", linewidth=2)

plt.xlabel("Mask area (%)")
plt.ylabel("Masked-region MAE")
plt.title("Masked-region MAE vs mask size")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## LoRA localization check

This test verifies whether the LoRA adapter has a larger effect inside the masked region than on the preserved background. A well-behaved inpainting LoRA should alter the masked region more than the unmasked area.


In [ ]:
masked_diffs = []
unmasked_diffs = []
for index, pair in enumerate(pairs):
    base_img = np.asarray(
        Image.open(run_dirs["base"] / f"{index:04d}.png").convert("RGB"),
        dtype=np.float32,
    )
    lora_img = np.asarray(
        Image.open(run_dirs["lora"] / f"{index:04d}.png").convert("RGB"),
        dtype=np.float32,
    )
    mask = np.asarray(load_mask(pair.mask_path), dtype=np.uint8) > 0
    mask_3d = np.broadcast_to(mask[..., None], base_img.shape)

    diff = np.abs(base_img - lora_img)
    masked_diffs.append(float(diff[mask_3d].mean()))
    unmasked_diffs.append(float(diff[~mask_3d].mean()))

print(f"Average base-vs-LoRA diff inside mask: {np.mean(masked_diffs):.2f}")
print(f"Average base-vs-LoRA diff outside mask: {np.mean(unmasked_diffs):.2f}")
print(
    "LoRA changes are",
    "more concentrated in the mask"
    if np.mean(masked_diffs) > np.mean(unmasked_diffs)
    else "not more concentrated in the mask",
)

In [ ]:
weights = Inception_V3_Weights.IMAGENET1K_V1
inception_preprocess = weights.transforms()

# Replacing the classifier with Identity exposes the 2048-dimensional pool3 features used by FID.
fid_model = inception_v3(weights=weights, transform_input=False)
fid_model.fc = torch.nn.Identity()
fid_model.eval().to(device)


@torch.no_grad()
def extract_inception_features(
    image_paths: list[Path], batch_size: int = BATCH_SIZE
) -> np.ndarray:
    """Convert images into Inception feature vectors for FID statistics."""
    features: list[np.ndarray] = []
    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start : start + batch_size]
        batch = torch.stack(
            [
                inception_preprocess(Image.open(path).convert("RGB"))
                for path in batch_paths
            ]
        ).to(device)
        batch_features = fid_model(batch)
        features.append(batch_features.detach().cpu().numpy())
    return np.concatenate(features, axis=0)


def feature_stats(features: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Compute mean and covariance for one image distribution."""
    if features.shape[0] < 2:
        raise ValueError(
            "FID needs at least two images per distribution; increase NUM_SAMPLES."
        )
    return features.mean(axis=0), np.cov(features, rowvar=False)


def fid_from_features(
    real_features: np.ndarray, generated_features: np.ndarray, eps: float = 1e-6
) -> float:
    """Compute Frechet Inception Distance from two sets of Inception features."""
    mu_real, sigma_real = feature_stats(real_features)
    mu_generated, sigma_generated = feature_stats(generated_features)

    diff = mu_real - mu_generated
    covmean, _ = linalg.sqrtm(sigma_real @ sigma_generated, disp=False)

    # Numerical issues can make sqrtm return non-finite or tiny imaginary values.
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma_real.shape[0]) * eps
        covmean = linalg.sqrtm((sigma_real + offset) @ (sigma_generated + offset))
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff.dot(diff) + np.trace(sigma_real + sigma_generated - 2.0 * covmean)
    return float(fid)


real_features = extract_inception_features(real_paths)
base_features = extract_inception_features(base_paths)
lora_features = extract_inception_features(lora_paths)

base_fid = fid_from_features(real_features, base_features)
lora_fid = fid_from_features(real_features, lora_features)

scores = {
    "base_fid": base_fid,
    "lora_fid": lora_fid,
    "lower_is_better": True,
    "num_samples": len(real_paths),
    "dataset_dir": str(DATASET_DIR),
    "test_mask_dir": str(TEST_MASK_DIR),
    "lora_dir": str(LORA_DIR),
    "lora_scale": LORA_SCALE,
    "seed": SEED,
    "prompt": PROMPT,
    "negative_prompt": NEGATIVE_PROMPT,
}

(OUTPUT_DIR / "fid_scores.json").write_text(
    json.dumps(scores, indent=2) + "\n", encoding="utf-8"
)
scores

In [ ]:
# Show a compact visual audit: original, mask, base output, and LoRA output for each evaluated sample.
rows = len(pairs)
fig, axes = plt.subplots(rows, 4, figsize=(12, max(3, rows * 3)))
if rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row, pair in enumerate(pairs):
    panels = [
        (load_image(pair.image_path), "original"),
        (load_mask(pair.mask_path), "mask"),
        (Image.open(base_paths[row]).convert("RGB"), "base"),
        (Image.open(lora_paths[row]).convert("RGB"), "base + LoRA"),
    ]
    for col, (image, title) in enumerate(panels):
        axes[row, col].imshow(image, cmap="gray" if title == "mask" else None)
        axes[row, col].set_title(title)
        axes[row, col].axis("off")

fig.suptitle(f"FID: base={base_fid:.2f}, base+LoRA={lora_fid:.2f} (lower is better)")
plt.tight_layout()
plt.show()